# Do two probes agree where their size ranges overlap?

Before two probes' size distributions are joined into one, they should agree in the size range
both of them measure. This notebook runs that check with
`arrayprobe_enhancements.probe_crossover.crossover_diagnostic`: second by second it integrates each
probe's concentration over the shared range, compares the two on the log ratio against its counting
uncertainty, and labels each second `agree`, `a_higher`, `b_higher`, `insufficient` or `missing`.
Ratios are quoted in dB. It runs in seconds on seeded random distributions and needs no data files.

**Before this:** [06_read_products](../1_getting_started/06_read_products.ipynb) (Reading the output files). **Related:** [01_cip_vs_2ds](01_cip_vs_2ds.ipynb). **Needs:** the enhancements package.

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/7_analysis_and_outputs/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## 1. Two synthetic probes of one leg

Probe A has fine bins and a small sample volume; probe B has coarse bins and a large one. Their bin
ladders share every edge from 0.15 to 1.5 mm. For the first 120 s both see the same cloud; for the
last 60 s probe B reads 3 dB high (a factor of $10^{0.3}$), the kind of excess a mis-set sample area
would give.

In [2]:
import numpy as np
from arrayprobe_enhancements.probe_crossover import ProbeSizeDistribution, crossover_diagnostic

rng = np.random.default_rng(46)
n_sec, n_bad, excess_db = 180, 60, 3.0
time = np.array([120000 + 100 * (s // 60) + s % 60 for s in range(n_sec)])   # HHMMSS
edges_a = np.round(np.arange(0.05, 1.5501, 0.05), 6)      # 30 fine bins
edges_b = np.round(np.arange(0.15, 3.0001, 0.15), 6)      # 19 coarse bins
gain_b = np.where(np.arange(n_sec) >= n_sec - n_bad, 10 ** (excess_db / 10), 1.0)

def probe(edges, sample_cm3, gain):
    mid, dD = 0.5 * (edges[:-1] + edges[1:]), np.diff(edges)
    expected = 0.5 * gain[:, None] * np.exp(-mid / 0.3)[None, :] * dD[None, :] * sample_cm3
    count = rng.poisson(expected).astype(float)
    return ProbeSizeDistribution(time=time, time_kind="hhmmss", bin_min=edges[:-1],
                                 bin_max=edges[1:], conc=count / sample_cm3, count=count,
                                 source_format="sd")

a = probe(edges_a, 400.0, np.ones(n_sec))
b = probe(edges_b, 4000.0, gain_b)
print(a.conc.shape, b.conc.shape)

(180, 30) (180, 19)


## 2. The per-second diagnostic

With no `size_range`, the comparison uses the widest range whose ends are edges of both ladders,
so every bin is wholly in or out of it. A second is tested only when each probe holds at least ten
effective particles in the range; a disagreement is flagged above three standard errors.

In [3]:
res = crossover_diagnostic(a, b)
ratio_db = 10 * np.log10(res.ratio)
print(f"size range {res.size_range[0]:.2f} to {res.size_range[1]:.2f} mm, edges aligned: {res.edges_aligned}")
print(res.counts_by_status())
print(f"flagged {res.n_flagged} of {res.n_tested} tested seconds; "
      f"counting noise alone would flag {res.expected_false_flags:.2f}")
good, bad = slice(0, n_sec - n_bad), slice(n_sec - n_bad, n_sec)
print(f"median A/B ratio, first {n_sec - n_bad} s: {np.median(ratio_db[good]):.2f} dB; "
      f"last {n_bad} s: {np.median(ratio_db[bad]):.2f} dB (built in: {-excess_db:.2f} dB)")
assert res.edges_aligned
assert abs(np.median(ratio_db[good])) < 0.5
assert abs(np.median(ratio_db[bad]) + excess_db) < 0.5
assert np.count_nonzero(res.status[bad] == "b_higher") > np.count_nonzero(res.status[good] == "b_higher")

size range 0.15 to 1.50 mm, edges aligned: True
{'missing': 0, 'insufficient': 0, 'agree': 125, 'a_higher': 1, 'b_higher': 54}
flagged 55 of 180 tested seconds; counting noise alone would flag 0.49
median A/B ratio, first 120 s: -0.01 dB; last 60 s: -3.04 dB (built in: -3.00 dB)


## 3. Averaging blocks of seconds

One second of a sparse range carries few particles, so a real offset is flagged in only some of
the seconds that carry it. `window_s` averages both probes over blocks of seconds before testing:
fewer tests, each with more particles behind it.

In [4]:
res10 = crossover_diagnostic(a, b, window_s=10)
print(res10.counts_by_status())
bad10 = slice(len(res10.status) - n_bad // 10, len(res10.status))
share_1 = np.mean(res.status[bad] == "b_higher")
share_10 = np.mean(res10.status[bad10] == "b_higher")
print(f"share of the offset stretch flagged b_higher: {share_1:.2f} per second, {share_10:.2f} per 10 s block")
assert share_10 >= share_1
assert not np.any(res10.status[: len(res10.status) - n_bad // 10] == "b_higher")

{'missing': 0, 'insufficient': 0, 'agree': 12, 'a_higher': 0, 'b_higher': 6}
share of the offset stretch flagged b_higher: 0.90 per second, 1.00 per 10 s block


## Summary

- `crossover_diagnostic` compares two probes over the size range both measure, second by second
  or over blocks of seconds, and labels each interval rather than averaging a disagreement away.
- The default range is snapped to shared bin edges, so no assumption is made about how particles
  spread inside a bin.
- `expected_false_flags` says how many flags counting noise alone would give; a count well above it
  is a real disagreement to resolve before the two probes are joined.
- Ratios between the probes are quoted in dB.